# 🎯 ABSA Vietnamese — Huấn luyện & Đánh giá 6 Mô hình trên Kaggle
---
**Luận văn Thạc sĩ**: Aspect-Based Sentiment Analysis cho tiếng Việt  
**Dữ liệu**: Đánh giá điện thoại Shopee (14,913 mẫu, 11 khía cạnh)  
**GPU**: Tesla T4 × 2 | **Env**: Miniconda + Python 3.10  

| # | Mô hình | Kiểu | Script | Config |
|---|---------|------|--------|--------|
| 1 | VisoBERT-MTL | Multi-Task | `train_visobert_mtl.py` | `config_visobert_mtl.yaml` |
| 2 | VisoBERT-STL | Two-Stage | `train_visobert_stl.py` | `config_visobert_stl.yaml` |
| 3 | PhoBERT-MTL | Multi-Task | `train_phobert_mtl.py` | `config_phobert_mtl.yaml` |
| 4 | PhoBERT-STL | Two-Stage | `train_phobert_stl.py` | `config_phobert_stl.yaml` |
| 5 | BiLSTM-MTL | Multi-Task | `train_bilstm_mtl.py` | `config_bilstm_mtl.yaml` |
| 6 | BiLSTM-STL | Two-Stage | `train_two_stage_bilstm.py` | `config_bilstm_stl.yaml` |

**Pipeline**: Setup → Clone → Data → W&B → Training (6 models) → Results → Error Analysis → Statistical Tests → Save → Summary

## 🔧 1. Setup & Dependencies
Cài Miniconda, accept ToS, tạo env `absa` (Python 3.10), cài thư viện.

In [ ]:
%%time
import subprocess, os, time

NOTEBOOK_START = time.time()

print('=' * 60)
print('📦 Cài đặt Miniconda...')
print('=' * 60)

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm -f Miniconda3-latest-Linux-x86_64.sh

CONDA = '/kaggle/working/miniconda/bin/conda'

# Accept Terms of Service (BẮT BUỘC trên Kaggle)
print('\n📜 Accept Anaconda ToS...')
!{CONDA} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!{CONDA} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Tạo environment absa
print('\n🐍 Tạo environment absa (Python 3.10)...')
!{CONDA} create -n absa python=3.10 -y

PYTHON = '/kaggle/working/miniconda/envs/absa/bin/python'
PIP = '/kaggle/working/miniconda/envs/absa/bin/pip'
print(f'\n✅ Python: {PYTHON}')
print(f'✅ Pip: {PIP}')

In [ ]:
%%time
print('=' * 60)
print('📚 Cài đặt thư viện...')
print('=' * 60)

PIP = '/kaggle/working/miniconda/envs/absa/bin/pip'
PYTHON = '/kaggle/working/miniconda/envs/absa/bin/python'

# PyTorch + CUDA
!{PIP} install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Các thư viện khác (bao gồm statsmodels cho McNemar test)
!{PIP} install -q transformers==4.44.2 accelerate>=0.20.0 datasets>=2.12.0 \
    scikit-learn>=1.3.0 pandas>=2.0.0 numpy>=1.24.0 \
    matplotlib>=3.7.0 seaborn>=0.12.0 pyyaml>=6.0 \
    tqdm>=4.65.0 sentencepiece>=0.1.99 protobuf>=3.20.0 wandb statsmodels

# Verify GPU
print('\n' + '=' * 60)
print('🖥️  Xác nhận GPU...')
print('=' * 60)

result = subprocess.run(
    [PYTHON, '-c',
     'import torch; '
     'print(f"PyTorch: {torch.__version__}"); '
     'print(f"CUDA available: {torch.cuda.is_available()}"); '
     'print(f"GPU count: {torch.cuda.device_count()}"); '
     '[print(f"  GPU {i}: {torch.cuda.get_device_name(i)}") for i in range(torch.cuda.device_count())]'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('⚠️ STDERR:', result.stderr[-1000:])

## 📂 2. Clone Repository

In [ ]:
%%time
import os

PROJECT_DIR = '/kaggle/working/ABSA-project'
BRANCH = 'multi-seed'  # ← MULTI-SEED: clone branch cụ thể

if os.path.exists(PROJECT_DIR):
    print(f'⚠️ {PROJECT_DIR} đã tồn tại, xóa và clone lại...')
    !rm -rf {PROJECT_DIR}

!git clone -b {BRANCH} https://github.com/hungtran3028/ABSA-project.git {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'\n📁 Working directory: {os.getcwd()}')
print(f'📌 Branch: {BRANCH}')

# Verify branch
!git branch --show-current

print('\n📋 Cấu trúc thư mục:')
for d in ['VisoBERT-MTL', 'VisoBERT-STL', 'phoBERT-MTL', 'PhoBERT-STL', 'BILSTM-MTL', 'BILSTM-STL', 'scripts']:
    exists = '✅' if os.path.isdir(d) else '❌'
    print(f'  {exists} {d}/')

import pandas as pd
df = pd.read_csv('dataset.csv')
print(f'\n📊 Dataset: {len(df)} mẫu, {len(df.columns)} cột')
print(f'   Cột: {list(df.columns)}')

## 📊 3. Data Preparation
Chạy `prepare_data_multilabel.py` để tạo train/validation/test splits.

In [ ]:
%%time
import subprocess, os, shutil

os.chdir('/kaggle/working/ABSA-project')
PYTHON = '/kaggle/working/miniconda/envs/absa/bin/python'

# Tạo clean env cho subprocess (fix MPLBACKEND conflict)
CLEAN_ENV = {k: v for k, v in os.environ.items() if k != 'MPLBACKEND'}
CLEAN_ENV['MPLBACKEND'] = 'Agg'
CLEAN_ENV['PYTHONPATH'] = '/kaggle/working/ABSA-project'

print('=' * 60)
print('📊 Chuẩn bị dữ liệu multi-label...')
print('=' * 60)

result = subprocess.run(
    [PYTHON, 'prepare_data_multilabel.py'],
    capture_output=True, text=True, timeout=300,
    env=CLEAN_ENV
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('⚠️ STDERR:', result.stderr[-2000:])

print('\n📋 Kiểm tra dữ liệu:')
for model_dir in ['VisoBERT-STL', 'PhoBERT-STL', 'BILSTM-MTL', 'BILSTM-STL']:
    data_dir = f'{model_dir}/data'
    if os.path.isdir(data_dir):
        csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
        print(f'  ✅ {data_dir}: {len(csv_files)} CSV files')
    else:
        print(f'  ❌ {data_dir}: chưa có')

data_copies = [
    ('VisoBERT-STL/data', 'phoBERT-MTL/data'),
    ('VisoBERT-STL/data', 'VisoBERT-MTL/data'),
]
for src, dst in data_copies:
    if os.path.isdir(src) and not os.path.isdir(dst):
        shutil.copytree(src, dst)
        print(f'  ✅ Copied: {src} → {dst}')
    elif os.path.isdir(dst):
        print(f'  ℹ️ Đã có: {dst}')

## 🔑 4. Weights & Biases Configuration

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = wandb_key
    print('✅ WANDB_API_KEY loaded from Kaggle Secrets')
except Exception as e:
    print(f'⚠️ Không lấy được WANDB_API_KEY: {e}')
    os.environ['WANDB_MODE'] = 'offline'
    print('📴 Wandb chạy ở chế độ offline')

os.environ['WANDB_PROJECT'] = 'ABSA-Vietnamese'
print(f'📊 Wandb project: {os.environ["WANDB_PROJECT"]}')

## 🏋️ 5. Huấn luyện 6 Mô hình
Chạy tuần tự: **VisoBERT-MTL → VisoBERT-STL → PhoBERT-MTL → PhoBERT-STL → BiLSTM-MTL → BiLSTM-STL**

Mỗi model chạy trong cell riêng, có error handling + memory cleanup.

In [ ]:
import yaml, re, os

# ╔══════════════════════════════════════════════════╗
# ║     🌱 MULTI-SEED CONFIG — ĐỔI Ở ĐÂY          ║
# ║     Session 1: CURRENT_SEED = 42                ║
# ║     Session 2: CURRENT_SEED = 213               ║
# ╚══════════════════════════════════════════════════╝
CURRENT_SEED = 42  # ← ĐỔI thành 213 cho session 2
SEED_SUFFIX = f"_seed{CURRENT_SEED}"

os.chdir('/kaggle/working/ABSA-project')

# ── 1. YAML seed keys ──
SEED_CONFIGS = {
    'VisoBERT-MTL/config_visobert_mtl.yaml': ['reproducibility', 'seed'],
    'VisoBERT-STL/config_visobert_stl.yaml': ['reproducibility', 'training_seed'],
    'phoBERT-MTL/config_phobert_mtl.yaml':   ['reproducibility', 'seed'],
    'PhoBERT-STL/config_phobert_stl.yaml':   ['reproducibility', 'training_seed'],
    'BILSTM-MTL/config_bilstm_mtl.yaml':     ['reproducibility', 'seed'],
    'BILSTM-STL/config_bilstm_stl.yaml':     ['reproducibility', 'seed'],
}

# ── 2. Output dir remapping ──
OUTPUT_CONFIGS = {
    'VisoBERT-MTL/config_visobert_mtl.yaml': {
        'paths.output_dir': f'VisoBERT-MTL/models/mtl{SEED_SUFFIX}',
        'paths.final_results_dir': f'VisoBERT-MTL/results{SEED_SUFFIX}',
    },
    'VisoBERT-STL/config_visobert_stl.yaml': {
        'paths.ad_output_dir': f'VisoBERT-STL/models/aspect_detection{SEED_SUFFIX}',
        'paths.sc_output_dir': f'VisoBERT-STL/models/sentiment_classification{SEED_SUFFIX}',
        'paths.final_results_dir': f'VisoBERT-STL/results/two_stage_training{SEED_SUFFIX}',
    },
    'phoBERT-MTL/config_phobert_mtl.yaml': {
        'paths.output_dir': f'phoBERT-MTL/models/mtl{SEED_SUFFIX}',
        'paths.final_results_dir': f'phoBERT-MTL/results{SEED_SUFFIX}',
    },
    'PhoBERT-STL/config_phobert_stl.yaml': {
        'paths.ad_output_dir': f'PhoBERT-STL/models/aspect_detection{SEED_SUFFIX}',
        'paths.sc_output_dir': f'PhoBERT-STL/models/sentiment_classification{SEED_SUFFIX}',
        'paths.final_results_dir': f'PhoBERT-STL/results/two_stage_training{SEED_SUFFIX}',
    },
    'BILSTM-MTL/config_bilstm_mtl.yaml': {
        'paths.output_dir': f'BILSTM-MTL/models/mtl{SEED_SUFFIX}',
        'paths.final_results_dir': f'BILSTM-MTL/results{SEED_SUFFIX}',
    },
    'BILSTM-STL/config_bilstm_stl.yaml': {
        'paths.ad_output_dir': f'BILSTM-STL/models/aspect_detection{SEED_SUFFIX}',
        'paths.sc_output_dir': f'BILSTM-STL/models/sentiment_classification{SEED_SUFFIX}',
        'paths.final_results_dir': f'BILSTM-STL/results/two_stage_training{SEED_SUFFIX}',
    },
}

# ── 3. WandB patch map ──
WANDB_PATCH_MAP = {
    'VisoBERT-MTL/train_visobert_mtl.py': 'ViSoBERT-MTL',
    'VisoBERT-STL/train_visobert_stl.py': 'ViSoBERT-STL',
    'phoBERT-MTL/train_phobert_mtl.py':   'PhoBERT-MTL',
    'PhoBERT-STL/train_phobert_stl.py':   'PhoBERT-STL',
    'BILSTM-MTL/train_bilstm_mtl.py':     'BiLSTM-MTL',
    'BILSTM-STL/train_two_stage_bilstm.py':'BiLSTM-STL',
}

# ── Helper functions ──
def set_nested(cfg, keys, value):
    obj = cfg
    for k in keys[:-1]:
        obj = obj[k]
    obj[keys[-1]] = value

def patch_wandb(script_path, model_name):
    with open(script_path, 'r', encoding='utf-8') as f:
        content = f.read()
    # name: add seed to run name
    old_name = f'name=f"{model_name}_'
    new_name = f'name=f"{model_name}_seed{CURRENT_SEED}_'
    content = content.replace(old_name, new_name)
    # group: for cross-seed comparison
    if 'group=' not in content:
        content = content.replace(new_name, f'group="{model_name}",\n            {new_name}')
    # tags: append seed tag
    content = re.sub(r'(tags=\[.*?)\]', rf'\1, "seed{CURRENT_SEED}"]', content)
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'  🔗 wandb: {model_name} → name/group/tag patched')

# ── APPLY ──
print(f'🌱 MULTI-SEED: Applying seed={CURRENT_SEED}')
print(f'   Output suffix: {SEED_SUFFIX}')
print()

# Patch YAML configs
for cfg_path, key_path in SEED_CONFIGS.items():
    with open(cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
    set_nested(cfg, key_path, CURRENT_SEED)
    if cfg_path in OUTPUT_CONFIGS:
        for dotkey, val in OUTPUT_CONFIGS[cfg_path].items():
            set_nested(cfg, dotkey.split('.'), val)
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print(f'  ✅ {cfg_path}: seed={CURRENT_SEED}')

# Patch wandb naming
print()
for script, model in WANDB_PATCH_MAP.items():
    patch_wandb(script, model)

print(f'\n✅ Tất cả configs + wandb đã patch cho seed={CURRENT_SEED}')

In [ ]:
import subprocess, os, gc, time, traceback

os.chdir('/kaggle/working/ABSA-project')
PYTHON = '/kaggle/working/miniconda/envs/absa/bin/python'

# FIX: Kaggle sets MPLBACKEND=module://matplotlib_inline.backend_inline
# nhưng env Miniconda không có matplotlib_inline → crash
# Override thành 'Agg' (non-interactive backend) cho tất cả subprocess
CLEAN_ENV = {k: v for k, v in os.environ.items() if k != 'MPLBACKEND'}
CLEAN_ENV['MPLBACKEND'] = 'Agg'
CLEAN_ENV['PYTHONPATH'] = '/kaggle/working/ABSA-project'

training_results = {}

def train_model(name, script, config, timeout=7200):
    """Huấn luyện 1 model với error handling."""
    print('=' * 70)
    print(f'🏋️ BẮT ĐẦU: {name}')
    print(f'   Script: {script}')
    print(f'   Config: {config}')
    print(f'   Timeout: {timeout}s ({timeout//60} phút)')
    print('=' * 70)
    start = time.time()
    try:
        result = subprocess.run(
            [PYTHON, script, '--config', config],
            capture_output=True, text=True, timeout=timeout,
            cwd='/kaggle/working/ABSA-project',
            env=CLEAN_ENV
        )
        elapsed = time.time() - start
        if result.stdout:
            lines = result.stdout.strip().split('\n')
            print(f'\n📝 Output (cuối {min(80, len(lines))} dòng):')
            for line in lines[-80:]:
                print(f'  {line}')
        if result.returncode == 0:
            print(f'\n✅ {name} — THÀNH CÔNG ({elapsed:.0f}s = {elapsed/60:.1f} phút)')
            training_results[name] = {'status': '✅ OK', 'time': elapsed}
        else:
            print(f'\n❌ {name} — LỖI (code: {result.returncode})')
            if result.stderr:
                print(result.stderr[-2000:])
            training_results[name] = {'status': '❌ FAIL', 'time': elapsed}
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start
        print(f'\n⏰ {name} — TIMEOUT sau {timeout}s')
        training_results[name] = {'status': '⏰ TIMEOUT', 'time': elapsed}
    except Exception as e:
        elapsed = time.time() - start
        print(f'\n💥 {name} — EXCEPTION: {e}')
        traceback.print_exc()
        training_results[name] = {'status': f'💥 {str(e)[:50]}', 'time': elapsed}
    finally:
        try:
            import torch
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except: pass
        gc.collect()
        print('🧹 GPU memory cleared\n')

print('✅ Helper function train_model() đã sẵn sàng.')
print(f'✅ MPLBACKEND = {CLEAN_ENV.get("MPLBACKEND", "not set")}')

### 🏋️ 5.1. VisoBERT-MTL

In [ ]:
%%time
train_model('VisoBERT-MTL', 'VisoBERT-MTL/train_visobert_mtl.py', 'VisoBERT-MTL/config_visobert_mtl.yaml')

### 🏋️ 5.2. VisoBERT-STL

In [ ]:
%%time
train_model('VisoBERT-STL', 'VisoBERT-STL/train_visobert_stl.py', 'VisoBERT-STL/config_visobert_stl.yaml')

### 🏋️ 5.3. PhoBERT-MTL

In [ ]:
%%time
train_model('PhoBERT-MTL', 'phoBERT-MTL/train_phobert_mtl.py', 'phoBERT-MTL/config_phobert_mtl.yaml')

### 🏋️ 5.4. PhoBERT-STL

In [ ]:
%%time
train_model('PhoBERT-STL', 'PhoBERT-STL/train_phobert_stl.py', 'PhoBERT-STL/config_phobert_stl.yaml')

### 🏋️ 5.5. BiLSTM-MTL

In [ ]:
%%time
train_model('BiLSTM-MTL', 'BILSTM-MTL/train_bilstm_mtl.py', 'BILSTM-MTL/config_bilstm_mtl.yaml')

### 🏋️ 5.6. BiLSTM-STL

In [ ]:
%%time
train_model('BiLSTM-STL', 'BILSTM-STL/train_two_stage_bilstm.py', 'BILSTM-STL/config_bilstm_stl.yaml')

## 📈 6. Tổng hợp Kết quả Training

In [ ]:
import os

os.chdir('/kaggle/working/ABSA-project')

S = SEED_SUFFIX  # From seed config cell

result_files = {
    'VisoBERT-MTL': [f'VisoBERT-MTL/models/mtl{S}/final_report.txt', f'VisoBERT-MTL/results{S}/final_report.txt'],
    'VisoBERT-STL': [f'VisoBERT-STL/results/two_stage_training{S}/final_report.txt'],
    'PhoBERT-MTL':  [f'phoBERT-MTL/models/mtl{S}/final_report.txt', f'phoBERT-MTL/results{S}/final_report.txt'],
    'PhoBERT-STL':  [f'PhoBERT-STL/results/two_stage_training{S}/final_report.txt'],
    'BiLSTM-MTL':   [f'BILSTM-MTL/models/mtl{S}/final_report.txt', f'BILSTM-MTL/results{S}/final_report.txt'],
    'BiLSTM-STL':   [f'BILSTM-STL/results/two_stage_training{S}/final_report.txt'],
}

print('=' * 80)
print(f'TỔNG HỢP KẾT QUẢ HUẤN LUYỆN 6 MÔ HÌNH ABSA (seed={CURRENT_SEED})')
print('=' * 80)
print()

for name, paths in result_files.items():
    found = False
    for path in paths:
        if os.path.isfile(path):
            print(f'📄 {name} — {path}')
            print('-' * 60)
            with open(path, 'r') as f:
                print(f.read())
            print()
            found = True
            break
    if not found:
        print(f'⚠️ {name} — Không tìm thấy file kết quả')
        print()

print('=' * 80)
print('TRAINING STATUS')
print('=' * 80)
print(f'{"Model":<20} {"Status":<15} {"Thời gian":<15}')
print('-' * 50)
for name, info in training_results.items():
    t = info['time']
    time_str = f'{t/60:.1f} phút' if t < 3600 else f'{t/3600:.1f} giờ'
    print(f'{name:<20} {info["status"]:<15} {time_str:<15}')
print('=' * 80)

## 🔬 7. Error Analysis
Confusion matrices, per-aspect F1, biểu đồ so sánh mô hình.

Chạy `scripts/run_error_analysis_all.py` → output trong `error_analysis_results/`.

In [ ]:
%%time
import subprocess, os, glob

os.chdir('/kaggle/working/ABSA-project')
PYTHON = '/kaggle/working/miniconda/envs/absa/bin/python'

print('=' * 70)
print('🔬 PHÂN TÍCH LỖI (Error Analysis) — 6 mô hình')
print('=' * 70)

try:
    result = subprocess.run(
        [PYTHON, 'scripts/run_error_analysis_all.py'],
        capture_output=True, text=True, timeout=600,
        cwd='/kaggle/working/ABSA-project',
        env=CLEAN_ENV
    )
    print(result.stdout[-5000:])
    if result.returncode != 0:
        print('⚠️ STDERR:', result.stderr[-2000:])
    else:
        print('\n✅ Error Analysis hoàn tất!')
except Exception as e:
    print(f'❌ Error Analysis thất bại: {e}')

ea_files = glob.glob('error_analysis_results/**/*', recursive=True)
if ea_files:
    print(f'\n📁 Tạo được {len(ea_files)} files trong error_analysis_results/')
    for f in sorted(ea_files)[:30]:
        size = os.path.getsize(f) if os.path.isfile(f) else 0
        print(f'  📄 {f} ({size/1024:.1f} KB)' if size > 0 else f'  📂 {f}/')
else:
    print('\n⚠️ Không tìm thấy output error analysis')

## 📊 8. Kiểm định Thống kê & Bảng Luận văn
- **McNemar's test**: MTL vs STL, ViSoBERT vs PhoBERT, Transformer vs BiLSTM
- **Thesis tables**: bảng LaTeX cho Chapter 4

In [ ]:
%%time
import subprocess, os

os.chdir('/kaggle/working/ABSA-project')
PYTHON = '/kaggle/working/miniconda/envs/absa/bin/python'

# --- McNemar's Test ---
print('=' * 70)
print('📊 MCNEMAR TEST — So sánh thống kê giữa các mô hình')
print('   H1: MTL vs STL | H2: ViSoBERT vs PhoBERT | H3: Transformer vs BiLSTM')
print('=' * 70)

try:
    result = subprocess.run(
        [PYTHON, 'scripts/run_mcnemar_test.py'],
        capture_output=True, text=True, timeout=300,
        cwd='/kaggle/working/ABSA-project',
        env=CLEAN_ENV
    )
    print(result.stdout)
    if result.returncode != 0:
        print('⚠️ STDERR:', result.stderr[-2000:])
    else:
        print('✅ McNemar Test hoàn tất!')
except Exception as e:
    print(f'❌ McNemar Test thất bại: {e}')

# --- Thesis Tables (LaTeX) ---
print('\n' + '=' * 70)
print('📋 BẢNG LUẬN VĂN — LaTeX format')
print('=' * 70)

try:
    result = subprocess.run(
        [PYTHON, 'scripts/generate_thesis_tables.py'],
        capture_output=True, text=True, timeout=300,
        cwd='/kaggle/working/ABSA-project',
        env=CLEAN_ENV
    )
    print(result.stdout)
    if result.returncode != 0:
        print('⚠️ STDERR:', result.stderr[-2000:])
    else:
        print('✅ Thesis Tables hoàn tất!')
except Exception as e:
    print(f'❌ Thesis Tables thất bại: {e}')

## 💾 9. Lưu Kết quả

In [ ]:
%%time
import shutil, os

os.chdir('/kaggle/working/ABSA-project')
OUTPUT_DIR = '/kaggle/working/ABSA-results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

S = SEED_SUFFIX  # From seed config cell

copy_map = {
    'VisoBERT-MTL': [f'VisoBERT-MTL/models/mtl{S}', f'VisoBERT-MTL/results{S}'],
    'VisoBERT-STL': [f'VisoBERT-STL/models/aspect_detection{S}', f'VisoBERT-STL/models/sentiment_classification{S}', f'VisoBERT-STL/results'],
    'PhoBERT-MTL':  [f'phoBERT-MTL/models/mtl{S}', f'phoBERT-MTL/results{S}'],
    'PhoBERT-STL':  [f'PhoBERT-STL/models/aspect_detection{S}', f'PhoBERT-STL/models/sentiment_classification{S}', f'PhoBERT-STL/results'],
    'BiLSTM-MTL':   [f'BILSTM-MTL/models/mtl{S}', f'BILSTM-MTL/results{S}'],
    'BiLSTM-STL':   [f'BILSTM-STL/models/aspect_detection{S}', f'BILSTM-STL/models/sentiment_classification{S}', f'BILSTM-STL/results'],
}

print('=' * 70)
print(f'Moving results to {OUTPUT_DIR} (seed={CURRENT_SEED})')
print('=' * 70)

for model_name, src_dirs in copy_map.items():
    dst_base = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(dst_base, exist_ok=True)
    for src_dir in src_dirs:
        if os.path.isdir(src_dir):
            dst = os.path.join(dst_base, os.path.basename(src_dir))
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.move(src_dir, dst)
            print(f'  ✅ {src_dir} → {dst}')
        else:
            print(f'  ⚠️ Not found: {src_dir}')

# Move error analysis results
ea_src = 'error_analysis_results'
ea_dst = os.path.join(OUTPUT_DIR, 'error_analysis_results')
if os.path.isdir(ea_src):
    if os.path.exists(ea_dst):
        shutil.rmtree(ea_dst)
    shutil.move(ea_src, ea_dst)
    print(f'  ✅ {ea_src} → {ea_dst}')

# Move wandb logs
wandb_src = 'wandb'
wandb_dst = os.path.join(OUTPUT_DIR, 'wandb')
if os.path.isdir(wandb_src):
    if os.path.exists(wandb_dst):
        shutil.rmtree(wandb_dst)
    shutil.move(wandb_src, wandb_dst)
    print(f'  ✅ {wandb_src} → {wandb_dst}')

# Total size
total_size = 0
total_files = 0
for dirpath, dirnames, filenames in os.walk(OUTPUT_DIR):
    for f in filenames:
        total_size += os.path.getsize(os.path.join(dirpath, f))
        total_files += 1

print(f'\n📦 Total: {total_files} files, {total_size / (1024*1024):.1f} MB')
print(f'📂 Output: {OUTPUT_DIR}')


## ✅ 10. Tổng kết

In [ ]:
import time

total_elapsed = time.time() - NOTEBOOK_START

print('=' * 70)
print('✅ TỔNG KẾT NOTEBOOK')
print('=' * 70)

success = sum(1 for v in training_results.values() if 'OK' in v['status'])
failed = len(training_results) - success

print(f'\n⏱️  Tổng thời gian: {total_elapsed/3600:.1f} giờ ({total_elapsed/60:.0f} phút)')
print(f'📊 Models thành công: {success}/6')
if failed > 0:
    print(f'❌ Models thất bại: {failed}/6')

print(f'\n{"Model":<20} {"Status":<15} {"Thời gian":<15}')
print('-' * 50)
for name, info in training_results.items():
    t = info['time']
    time_str = f'{t/60:.1f} phút' if t < 3600 else f'{t/3600:.1f} giờ'
    print(f'{name:<20} {info["status"]:<15} {time_str:<15}')

print('\n' + '=' * 70)
print('🎓 ABSA Vietnamese — Huấn luyện & Đánh giá hoàn tất!')
print('=' * 70)